In [1]:
import pandas as pd
import sqlite3

# Load our two clean files
jobs = pd.read_csv("../data/processed/jobs_clean.csv")
job_skills = pd.read_csv("../data/processed/job_skills.csv")

# Create (or open) a database file
conn = sqlite3.connect("../data/processed/jobs.db")

# Put each table into the database
jobs.to_sql("jobs", conn, if_exists="replace", index=False)
job_skills.to_sql("job_skills", conn, if_exists="replace", index=False)

# Our first SQL query: count the jobs
pd.read_sql("SELECT COUNT(*) AS total_jobs FROM jobs", conn)

,total_jobs
0,2314


In [2]:
query = """
SELECT city,
       COUNT(*) AS total_jobs
FROM jobs
WHERE city != 'India (unspecified)'
GROUP BY city
ORDER BY total_jobs DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,city,total_jobs
0,Bangalore,665
1,Hyderabad,251
2,Pune,156
3,Mumbai,138
4,Chennai,113
5,Noida,61
6,Delhi,41
7,Ahmedabad,33
8,Gurgaon,30
9,Navi Mumbai,20


In [3]:
query = """
SELECT role,
       COUNT(salary_avg_lakh) AS jobs_with_salary,
       ROUND(AVG(salary_avg_lakh), 1) AS avg_salary_lakh
FROM jobs
WHERE role != 'Other'
GROUP BY role
ORDER BY avg_salary_lakh DESC
"""

pd.read_sql(query, conn)

,role,jobs_with_salary,avg_salary_lakh
0,AI Engineer,76,17.4
1,ML Engineer,72,16.2
2,Data Scientist,114,15.5
3,Data Engineer,63,14.8
4,Data Analyst,97,9.5
5,Business Analyst,112,8.9


In [4]:
query = """
SELECT s.skill,
       COUNT(*) AS jobs
FROM job_skills AS s
JOIN jobs AS j
     ON s.job_id = j.job_id
WHERE j.role = 'Data Analyst'
GROUP BY s.skill
ORDER BY jobs DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,skill,jobs
0,sql,81
1,python,44
2,stakeholder,42
3,excel,34
4,power bi,31
5,communication,24
6,tableau,23
7,statistics,8
8,snowflake,7
9,spark,4


In [5]:
query = """
SELECT role,
       COUNT(*) AS total_jobs,
       SUM(CASE WHEN seniority IN ('Intern', 'Junior') THEN 1 ELSE 0 END) AS entry_level,
       SUM(CASE WHEN seniority IN ('Senior', 'Lead / Principal') THEN 1 ELSE 0 END) AS senior_level
FROM jobs
WHERE role != 'Other'
GROUP BY role
ORDER BY senior_level DESC
"""

pd.read_sql(query, conn)

,role,total_jobs,entry_level,senior_level
0,AI Engineer,431,14,174
1,ML Engineer,335,7,167
2,Data Engineer,381,1,94
3,Data Analyst,405,13,88
4,Data Scientist,309,7,42
5,Business Analyst,348,0,12


In [6]:
#Which cities pay the most
query = """
SELECT city,
       COUNT(salary_avg_lakh) AS jobs_with_salary,
       ROUND(AVG(salary_avg_lakh), 1) AS avg_salary_lakh
FROM jobs
WHERE city != 'India (unspecified)'
GROUP BY city
HAVING COUNT(salary_avg_lakh) >= 15
ORDER BY avg_salary_lakh DESC
"""

pd.read_sql(query, conn)

,city,jobs_with_salary,avg_salary_lakh
0,Pune,41,16.0
1,Hyderabad,40,15.8
2,Bangalore,150,15.8
3,Chennai,19,14.2
4,Mumbai,53,12.1
5,Noida,16,11.7
6,Delhi,21,8.3
